In [ ]:
from google.colab import drive, files, output
import shutil

drive.mount('/content/drive')

client_secrets_path = '/content/drive/MyDrive/json/.client_secrets.json'
credentials_path = '/content/drive/MyDrive/json/.youtube-upload-credentials.json'
shutil.copy(client_secrets_path, './client_secrets.json')
shutil.copy(credentials_path, './credentials.json')
client_secrets = './client_secrets.json'
credentials = './credentials.json'

In [ ]:
!pip install PyCryptodome==3.17.0
!pip install yt_dlp
!pip install git+https://github.com/tokland/youtube-upload.git
output.clear()

In [5]:
# built-in
import os
import cv2
import base64
import uuid
import numpy as np
import tempfile
import subprocess
import concurrent.futures

# 3rd party
import yt_dlp
from Crypto.Cipher import AES
from youtube_upload import auth, lib, upload_video

In [10]:
def quotient_remainder(divident, divsor):
    return divident // divsor, divident % divsor


def color_value(x):
    return x*255


def normal(x):
    return x/255


def encrypt_data_aes(data: bytes, key: bytes) -> bytes:
    cipher = AES.new(key, AES.MODE_EAX)
    ciphertext, tag = cipher.encrypt_and_digest(data)
    return base64.urlsafe_b64encode(cipher.nonce + tag + ciphertext)


def decrypt_data_aes(data: bytes, key: bytes) -> bytes:
    raw = base64.urlsafe_b64decode(data)
    nonce, tag, ciphertext = raw[:16], raw[16:32], raw[32:]

    cipher = AES.new(key, AES.MODE_EAX, nonce)
    clear_data = cipher.decrypt_and_verify(ciphertext, tag)
    return clear_data


def prepare_frame(args):
    frame_bytes, num_rows_per_frame, num_cols_per_frame, color_value, size = args
    frame_bits = np.unpackbits(frame_bytes)
    frame = color_value(frame_bits).reshape(num_rows_per_frame, num_cols_per_frame, 3).astype(np.uint8)
    newimg = cv2.resize(frame, size, interpolation=cv2.INTER_AREA)
    return newimg


def detect_ffmpeg_gpu_encoder():
    # Check for NVIDIA GPU
    try:
        result = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True)
        encoders = result.stdout
        if 'hevc_nvenc' in encoders:
            return 'hevc_nvenc'
        elif 'hevc_qsv' in encoders:
            return 'hevc_qsv'
        elif 'hevc_videotoolbox' in encoders:
            return 'hevc_videotoolbox'
    except Exception:
        pass
    return 'libx265'  # CPU fallback


def encode(infile_path, outvideo_path, encrypt, key,
           fps=20, num_cols_per_frame=64, num_rows_per_frame=36):
    with open(infile_path, 'rb') as fd:
        raw_data_bytes = fd.read()
    if encrypt:
        raw_data_bytes = encrypt_data_aes(raw_data_bytes, key)
    data_bytes = np.frombuffer(raw_data_bytes, dtype=np.uint8)
    len_of_data = len(data_bytes)
    num_bytes_per_row = int(num_cols_per_frame * 3 / 8)
    num_bytes_per_frame = num_bytes_per_row * num_rows_per_frame

    len_bytes = np.frombuffer(len_of_data.to_bytes(4, byteorder='big'), dtype=np.uint8)
    total_data = [len_bytes, data_bytes]

    (num_frames, num_leftover_bytes) = quotient_remainder(4 + len_of_data, num_bytes_per_frame)

    if num_leftover_bytes > 0:
        num_bytes_last_frame_padding = num_bytes_per_frame - num_leftover_bytes
        padding_bytes = np.zeros(num_bytes_last_frame_padding, dtype=np.uint8)
        total_data.append(padding_bytes)
        num_frames += 1

    data_bytes = np.concatenate(total_data)

    size = (num_cols_per_frame * 20, num_rows_per_frame * 20)

    args_list = [
        (
            data_bytes[i * num_bytes_per_frame: (i + 1) * num_bytes_per_frame],
            num_rows_per_frame,
            num_cols_per_frame,
            color_value,
            size
        )
        for i in range(num_frames)
    ]

    # Create a temporary directory for PNG frames
    with tempfile.TemporaryDirectory() as temp_dir:
        # Save frames as PNG images in the temp directory
        with concurrent.futures.ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
            for idx, newimg in enumerate(executor.map(prepare_frame, args_list)):
                cv2.imwrite(os.path.join(temp_dir, f"frame_{idx:04d}.png"), newimg)

        # Detect GPU encoder or fallback to CPU
        encoder = detect_ffmpeg_gpu_encoder()
        print(f"Using FFmpeg encoder: {encoder}")

        # Run ffmpeg using images from the temp directory
        ffmpeg_cmd = (
            f'ffmpeg -y -framerate {fps} -i "{os.path.join(temp_dir, "frame_%04d.png")}" '
            f'-c:v {encoder} "{outvideo_path}"'
        )
        os.system(ffmpeg_cmd)


def process_frame(args):
    frame, step = args
    blocks = frame.reshape(frame.shape[0]//step, step, frame.shape[1]//step, step, 3)
    blocks = blocks.transpose(0,2,1,3,4).reshape(-1, step*step, 3)
    means = normal(blocks.mean(axis=1)).round().astype(np.uint8)
    return means


def decode(invideo_path, outfile_path, decrypt, key):
    step = 20
    cap = cv2.VideoCapture(invideo_path)
    data_bits_list = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        data_bits_list.append(process_frame((frame, step)))
    cap.release()
    data_bits = np.concatenate(data_bits_list).reshape(-1, 1)
    data_bytes = np.packbits(data_bits)
    len_of_data = int.from_bytes(data_bytes[:4], byteorder='big')
    data_bytes_retrieved = data_bytes[4:len_of_data+4].tobytes()
    if decrypt:
        data_bytes_retrieved = decrypt_data_aes(data_bytes_retrieved, key)
    with open(outfile_path, 'wb') as fd:
        fd.write(data_bytes_retrieved)


In [7]:
def get_youtube_upload_handler():
    """Return the API Youtube object."""
    # home = os.path.expanduser("~")
    # client_secrets = os.path.join(home, ".client_secrets.json")
    # credentials = os.path.join(home, ".youtube-upload-credentials.json")
    get_code_callback = (auth.console.get_code)
    return auth.get_resource(client_secrets, credentials,
                             get_code_callback=get_code_callback)


def upload_youtube_video(youtube, title, video_path):
    """Upload video with index (for split videos)."""
    u = lib.to_utf8
    title = u(title)

    request_body = {
        "snippet": {
            "title": title,
            "description": "",
            "categoryId": None,
            "tags": u(""),
            "defaultLanguage": None,
            "defaultAudioLanguage": None

        },
        "status": {
            "embeddable": True,
            "privacyStatus": "unlisted",
            "publishAt": None,
            "license": "youtube",
        },
        "recordingDetails": {
            "location": None,
            "recordingDate": None,
        },
    }

    return upload_video.upload(youtube, video_path, request_body)


def download_youtube_video(video_id, output_file_path):
    ydl_opts = {
        'outtmpl': output_file_path,
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4',
        'merge_output_format': 'mp4'
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([f"https://www.youtube.com/watch?v={video_id}"])


def youtube_upload(upload_file_path, encrypt, key, video_fps):
    video_path = tempfile.mktemp(".mp4")
    encode(upload_file_path, video_path, encrypt, key, video_fps)
    youtube = get_youtube_upload_handler()
    video_id = upload_youtube_video(youtube, f"DATA-{str(uuid.uuid4()).upper()}", video_path)
    os.remove(video_path)
    print(f"YouTube Video ID: {video_id}", f"YouTube: https://youtu.be/{video_id}", sep="\n")

def youtube_retrieve(video_id, output_file_path, decrypt, key):
    video_path = tempfile.mktemp(".mp4")
    print(video_path)
    download_youtube_video(video_id, video_path)
    decode(video_path, output_file_path, decrypt, key)


In [ ]:
# upload

uploaded = files.upload()
i = os.path.join("/content/", next(iter(uploaded.keys())))
encrypt = False
key = ""

key = str(key).encode("ascii")[:16] if encrypt and key is not None else ""
youtube_upload(i, encrypt, key, video_fps=20)


In [ ]:
# retrieve

video_id = input("Enter YouTube Video ID to retrieve: ")
output_file_path = input("Enter output file path to save the retrieved video: ")
decrypt = False
key = ""

key = str(key).encode("ascii")[:16] if encrypt and key is not None else ""
youtube_retrieve(video_id, output_file_path, decrypt, key)

In [11]:
# encode test
encode("1MB.jpg", "1MB.mp4", False, "")

Using FFmpeg encoder: hevc_videotoolbox


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex